
## A) RAG + FAISS Vector Store Pipeline 
## B) Tutoring Insights Agent (RAG + Tools + PDF) 

This notebook organizes a simple Retrieval-Augmented Generation (RAG) flow with:
- **Document loading** (JSON example included; extend as needed)
- **Chunking & Embeddings** with SentenceTransformers
- **FAISS** vector index build/load/query
- **Light RAG wrapper** that retrieves top chunks and summarizes with a Groq LLM
- **Insights Agent** analyzes tutoring transcripts to extract meaningful insights about a student’s understanding, learning progress, misconceptions, and next learning actions. Also saves it as pdf

> **Security note:**  Set environment variables via `.env` or your notebook environment.


In [1]:

from pathlib import Path
from typing import List, Any, Optional, Dict
import os
import pickle
import numpy as np

# LangChain / loaders / text splitters
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector store
import faiss

# LLM (Groq)
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables (optional if using a .env file)
load_dotenv()

DATA_DIR = "data"
PERSIST_DIR = "faiss_store"
EMBED_MODEL = "all-MiniLM-L6-v2"  # You can switch to larger models if desired
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5



## 1) Data Loading

This example demonstrates **JSON** loading using `JSONLoader` with a simple jq schema:
```jq
.[] | {role: .role, text: .text}
```
Adjust the schema to match your file format. You can extend the loader to handle PDFs, CSVs, etc., as needed.


In [2]:

def load_all_documents(data_dir: str) -> List[Any]:
    """
    Load supported files from the data directory and convert to LangChain document structure.
    Currently implemented: **JSON** (extend for PDF/TXT/CSV/Excel/Word).
    Expects an array of objects with `.role` and `.text` (adjust jq_schema as needed).
    """
    data_path = Path(data_dir).resolve()
    print(f"[DEBUG] Data path: {data_path}")
    documents = []

    # JSON (recursive)
    json_files = list(data_path.glob('**/*.json'))
    print(f"[DEBUG] Found {len(json_files)} JSON files: {[str(f) for f in json_files]}")

    for json_file in json_files:
        print(f"[DEBUG] Loading JSON: {json_file}")
        try:
            loader = JSONLoader(
                file_path=str(json_file),
                jq_schema=".[] | {role: .role, text: .text}",
                text_content=False
            )
            loaded = loader.load()
            print(f"[DEBUG] Loaded {len(loaded)} JSON docs from {json_file}")
            documents.extend(loaded)
        except Exception as e:
            print(f"[ERROR] Failed to load JSON {json_file}: {e}")

    print(f"[DEBUG] Total loaded documents: {len(documents)}")
    return documents

# Example: preview (safe even if no files present)
docs = load_all_documents(DATA_DIR)
print(f"Loaded {len(docs)} documents.")
print('Example document:', docs[0] if docs else None)


[DEBUG] Data path: /Users/abiramashree/Downloads/AI/rag_llm/data
[DEBUG] Found 2 JSON files: ['/Users/abiramashree/Downloads/AI/rag_llm/data/science_class.json', '/Users/abiramashree/Downloads/AI/rag_llm/data/math_class.json']
[DEBUG] Loading JSON: /Users/abiramashree/Downloads/AI/rag_llm/data/science_class.json
[DEBUG] Loaded 16 JSON docs from /Users/abiramashree/Downloads/AI/rag_llm/data/science_class.json
[DEBUG] Loading JSON: /Users/abiramashree/Downloads/AI/rag_llm/data/math_class.json
[DEBUG] Loaded 33 JSON docs from /Users/abiramashree/Downloads/AI/rag_llm/data/math_class.json
[DEBUG] Total loaded documents: 49
Loaded 49 documents.
Example document: page_content='{"role": "willy", "text": "Hi.\n"}' metadata={'source': '/Users/abiramashree/Downloads/AI/rag_llm/data/science_class.json', 'seq_num': 1}



## 2) Chunking & Embeddings
We chunk documents using `RecursiveCharacterTextSplitter`, then compute embeddings with a SentenceTransformers model.


In [3]:

class EmbeddingPipeline:
    def __init__(self, model_name: str = EMBED_MODEL, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)
        print(f"[INFO] Loaded embedding model: {model_name}")

    def chunk_documents(self, documents: List[Any]) -> List[Any]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(documents)
        print(f"[INFO] Split {len(documents)} documents into {len(chunks)} chunks.")
        return chunks

    def embed_chunks(self, chunks: List[Any]) -> np.ndarray:
        texts = [chunk.page_content for chunk in chunks]
        print(f"[INFO] Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"[INFO] Embeddings shape: {np.asarray(embeddings).shape}")
        return np.asarray(embeddings)

# Example usage (you can run these when you have data):
emb_pipe = EmbeddingPipeline()
chunks = emb_pipe.chunk_documents(docs)
embeddings = emb_pipe.embed_chunks(chunks)
print("[INFO] Example embedding vector length:", embeddings.shape[1] if len(embeddings) else None)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Split 49 documents into 49 chunks.
[INFO] Generating embeddings for 49 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Embeddings shape: (49, 384)
[INFO] Example embedding vector length: 384



## 3) FAISS Vector Store
Build, save, load, and query a FAISS index. Metadata stores chunk text for quick inspection.


In [4]:

class FaissVectorStore:
    def __init__(self, persist_dir: str = PERSIST_DIR, embedding_model: str = EMBED_MODEL, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
        self.persist_dir = persist_dir
        Path(self.persist_dir).mkdir(parents=True, exist_ok=True)
        self.index: Optional[faiss.Index] = None
        self.metadata: List[Dict[str, Any]] = []
        self.embedding_model = embedding_model
        self.model = SentenceTransformer(embedding_model)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        print(f"[INFO] Loaded embedding model (for queries): {embedding_model}")

    def build_from_documents(self, documents: List[Any]):
        print(f"[INFO] Building vector store from {len(documents)} raw documents...")
        emb_pipe = EmbeddingPipeline(model_name=self.embedding_model, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        chunks = emb_pipe.chunk_documents(documents)
        embeddings = emb_pipe.embed_chunks(chunks)
        metadatas = [{"text": chunk.page_content} for chunk in chunks]
        self.add_embeddings(np.array(embeddings).astype('float32'), metadatas)
        self.save()
        print(f"[INFO] Vector store built and saved to {self.persist_dir}")

    def add_embeddings(self, embeddings: np.ndarray, metadatas: Optional[List[Dict[str, Any]]] = None):
        if embeddings.ndim != 2:
            raise ValueError("Embeddings must be a 2D array of shape (n_vectors, dim).")
        dim = embeddings.shape[1]
        if self.index is None:
            self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)
        if metadatas:
            self.metadata.extend(metadatas)
        print(f"[INFO] Added {embeddings.shape[0]} vectors to Faiss index. Current total: {self.index.ntotal}")

    def save(self):
        if self.index is None:
            raise ValueError("No FAISS index to save.")
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        faiss.write_index(self.index, faiss_path)
        with open(meta_path, "wb") as f:
            pickle.dump(self.metadata, f)
        print(f"[INFO] Saved Faiss index and metadata to {self.persist_dir}")

    def load(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        if not (os.path.exists(faiss_path) and os.path.exists(meta_path)):
            raise FileNotFoundError(f"Missing index or metadata at {self.persist_dir}. Build first or check paths.")
        self.index = faiss.read_index(faiss_path)
        with open(meta_path, "rb") as f:
            self.metadata = pickle.load(f)
        print(f"[INFO] Loaded Faiss index and metadata from {self.persist_dir}")

    def search(self, query_embedding: np.ndarray, top_k: int = TOP_K):
        if self.index is None:
            raise ValueError("FAISS index is not loaded or built.")
        if query_embedding.ndim == 1:
            query_embedding = query_embedding.reshape(1, -1)
        D, I = self.index.search(query_embedding.astype('float32'), top_k)
        results = []
        # Gracefully handle if there are fewer vectors than top_k
        for idx, dist in zip(I[0], D[0]):
            if idx == -1:
                continue
            meta = self.metadata[idx] if 0 <= idx < len(self.metadata) else None
            results.append({"index": int(idx), "distance": float(dist), "metadata": meta})
        return results

    def query(self, query_text: str, top_k: int = TOP_K):
        print(f"[INFO] Querying vector store for: '{query_text}'")
        query_emb = self.model.encode([query_text]).astype('float32')
        return self.search(query_emb, top_k=top_k)

# Example guarded usage:
store = FaissVectorStore(PERSIST_DIR)
if len(docs) > 0:
    store.build_from_documents(docs)
store.load()
print(store.query("What are the three states of matter?", top_k=3))


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[INFO] Loaded embedding model (for queries): all-MiniLM-L6-v2
[INFO] Building vector store from 49 raw documents...
[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Split 49 documents into 49 chunks.
[INFO] Generating embeddings for 49 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Embeddings shape: (49, 384)
[INFO] Added 49 vectors to Faiss index. Current total: 49
[INFO] Saved Faiss index and metadata to faiss_store
[INFO] Vector store built and saved to faiss_store
[INFO] Loaded Faiss index and metadata from faiss_store
[INFO] Querying vector store for: 'What are the three states of matter?'
[{'index': 3, 'distance': 0.8391778469085693, 'metadata': {'text': '{"role": "assistant", "text": "Great! Matter is anything that takes up space and has mass. There are three main states of matter: solid, liquid, and gas.\\n\\nLet\'s begin with Step 1.\\n\\nFirst question: Can you name one example of a solid?"}'}}, {'index': 1, 'distance': 1.1207845211029053, 'metadata': {'text': '{"role": "assistant", "text": "Hey there! Welcome to today\'s science tutoring session! Today, we\\u2019re going to learn about the states of matter. Would you like me to start by explaining what they are?"}'}}, {'index': 15, 'distance': 1.2893387079238892, 'metadata': {'text': '{"role": "


## 4) RAG Search (Retrieve + Summarize)

`RAGSearch` wraps the vector store retrieval and then summarizes the top-k chunks with a Groq LLM.
- Set your **GROQ_API_KEY** in the environment (e.g., via `.env`).
- Choose an appropriate **model** (e.g., `llama-3.1-8b-instant`, `llama3-70b-8192`, etc.).


In [5]:

class RAGSearch:
    def __init__(
        self,
        persist_dir: str = PERSIST_DIR,
        embedding_model: str = EMBED_MODEL,
        llm_model: str = "llama-3.1-8b-instant",
    ):
        # Initialize / load vector store
        self.vectorstore = FaissVectorStore(persist_dir, embedding_model)
        faiss_path = os.path.join(persist_dir, "faiss.index")
        meta_path = os.path.join(persist_dir, "metadata.pkl")
        if os.path.exists(faiss_path) and os.path.exists(meta_path):
            self.vectorstore.load()
        else:
            docs = load_all_documents(DATA_DIR)
            if not docs:
                raise RuntimeError("No documents found to build the vector store. Add data to the 'data/' folder.")
            self.vectorstore.build_from_documents(docs)
        load_dotenv()
        # Initialize LLM (Groq)
        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise EnvironmentError("GROQ_API_KEY not found. Please set it in your environment or .env file.")
        self.llm = ChatGroq(groq_api_key=groq_api_key, model_name=llm_model)
        print(f"[INFO] Groq LLM initialized: {llm_model}")

    def search_and_summarize(self, query: str, top_k: int = TOP_K) -> str:
        results = self.vectorstore.query(query, top_k=top_k)
        texts = [r.get("metadata", {}).get("text", "") for r in results if r.get("metadata")]
        context = "\n\n".join([t for t in texts if t])

        if not context.strip():
            return "No relevant documents found."

        # Single-string prompt (ChatGroq supports various call patterns; using simple string here)
        prompt = (
            f"Summarize the following context for the query: '{query}'.\n\n"
            f"Context:\n{context}\n\n"
            "Return a concise, faithful summary."
        )
        response = self.llm.invoke(prompt)
        # .invoke returns an object with .content (for newer LC versions); handle both cases
        content = getattr(response, "content", None) or getattr(response, "text", None) or str(response)
        return content



## 5) Quick Demo for RAG retrieval and summary from LLM

Set `RUN_DEMO = True` to build/load, query, and summarize (requires data and `GROQ_API_KEY`).


In [6]:

RUN_DEMO = True

if RUN_DEMO:

    # RAG summarize
    rag = RAGSearch(persist_dir=PERSIST_DIR, embedding_model=EMBED_MODEL, llm_model="llama-3.1-8b-instant")
    print(rag.search_and_summarize("Did the student understand the three states of matter?", top_k=3))


[INFO] Loaded embedding model (for queries): all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
[INFO] Groq LLM initialized: llama-3.1-8b-instant
[INFO] Querying vector store for: 'Did the student understand the three states of matter?'
The student was learning about the three states of matter: solid, liquid, and gas. They provided examples and learned how these states change (e.g., liquid to solid through freezing).


## B) Tutoring Insights Agent (RAG + Tools + PDF) 

In [7]:
import os, json
from typing import List, Dict, Any, Optional
from pathlib import Path
from datetime import datetime

# ----------------- Pydantic schema -----------------
from pydantic import BaseModel, Field
from typing import Literal

class TutoringInsights(BaseModel):
    topics: List[str] = Field(default_factory=list)
    student_understanding: Literal["low","medium","high"] = "medium"
    misconceptions: List[str] = Field(default_factory=list)
    knowledge_gaps: List[str] = Field(default_factory=list)
    questions_to_ask: List[str] = Field(default_factory=list)
    suggested_activities: List[str] = Field(default_factory=list)
    next_best_prompts: List[str] = Field(default_factory=list)
    actionable_feedback: List[str] = Field(default_factory=list)
    summary: str = ""
    difficulty_rating: int = Field(3, ge=1, le=5)
    blooms_level: Literal["Remember","Understand","Apply","Analyze","Evaluate","Create"] = "Understand"
    sentiment: Literal["frustrated","neutral","confident","excited","confused"] = "neutral"
    key_quotes: List[str] = Field(default_factory=list)
    confidence: float = Field(0.6, ge=0, le=1)

DEFAULT_SYSTEM_PROMPT = (
    "You are a tutoring analytics coach. Read student–tutor transcripts and produce precise, "
    "actionable insights to plan the next step. Be diagnostic, concise, and concrete. "
    "Identify misconceptions and gaps; propose targeted questions and activities. "
    "Always return output that follows the requested format."
)

# ----------------- Tools -----------------
try:
    from langchain_core.tools import tool   # LC >= 0.2
except ImportError:
    from langchain.tools import tool        # fallback

# IMPORTANT: global `rag` must exist:
# rag = RAGSearch(persist_dir="faiss_store", embedding_model="all-MiniLM-L6-v2", llm_model="llama-3.1-8b-instant")

@tool("rag_search", return_direct=False)
def rag_search(query: str) -> str:
    """
    Retrieve top-k transcript chunks for the user's query from the FAISS-backed vector store.
    Input: a plain query string.
    Output: a single string with concatenated chunks, or [EMPTY] if none.
    """
    results = rag.vectorstore.query(query, top_k=5)
    texts = [r.get("metadata", {}).get("text", "") for r in results if r.get("metadata")]
    joined = "\n\n".join([t for t in texts if t])
    return joined if joined.strip() else "[EMPTY]"

# ---- PDF tool ----
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors

def _save_tutoring_insights_pdf_core(
    insights: Dict[str, Any],
    retrieved_chunks: Optional[List[str]] = None,
    output_path: str = "tutoring_insights.pdf",
) -> str:
    retrieved_chunks = retrieved_chunks or []
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("<b>Tutoring Session Insights Report</b>", styles["Title"]))
    story.append(Paragraph(datetime.now().strftime("%B %d, %Y %H:%M"), styles["Normal"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("<b>Summary</b>", styles["Heading2"]))
    story.append(Paragraph(insights.get("summary", "No summary."), styles["BodyText"]))
    story.append(Spacer(1, 12))

    core_data = [
        ["Student Understanding", insights.get("student_understanding", "")],
        ["Difficulty Rating", str(insights.get("difficulty_rating", ""))],
        ["Bloom’s Level", insights.get("blooms_level", "")],
        ["Sentiment", insights.get("sentiment", "")],
        ["Confidence", f"{float(insights.get('confidence', 0.0)):.2f}"],
    ]
    table = Table(core_data, hAlign="LEFT")
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
        ('GRID', (0,0), (-1,-1), 0.25, colors.grey),
    ]))
    story.append(Paragraph("<b>Session Metrics</b>", styles["Heading2"]))
    story.append(table)
    story.append(Spacer(1, 12))

    for title in [
        "topics", "misconceptions", "knowledge_gaps",
        "questions_to_ask", "suggested_activities",
        "next_best_prompts", "actionable_feedback",
        "key_quotes",
    ]:
        vals = insights.get(title, [])
        if vals:
            story.append(Paragraph(f"<b>{title.replace('_',' ').title()}</b>", styles["Heading3"]))
            for item in vals:
                story.append(Paragraph(f"• {item}", styles["BodyText"]))
            story.append(Spacer(1, 6))

    if retrieved_chunks:
        story.append(Spacer(1, 12))
        story.append(Paragraph("<b>Retrieved Transcript Chunks</b>", styles["Heading2"]))
        for i, ch in enumerate(retrieved_chunks[:5], 1):
            snippet = ch[:500] + ("..." if len(ch) > 500 else "")
            story.append(Paragraph(f"[{i}] {snippet}", styles["Code"]))
            story.append(Spacer(1, 4))

    out_path = Path(output_path).resolve()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc = SimpleDocTemplate(str(out_path), pagesize=A4)
    doc.build(story)
    return str(out_path)

@tool("save_tutoring_insights_pdf", return_direct=False)
def save_tutoring_insights_pdf(payload: str) -> str:
    """
    Save TutoringInsights to a PDF.

    Args (JSON string):
      - insights: dict (TutoringInsights fields)
      - retrieved_chunks: list[str] (optional)
      - output_path: str (optional; default 'tutoring_insights.pdf')
    Returns: absolute path string or an error.
    """
    try:
        data = json.loads(payload)
        insights = data.get("insights")
        if not isinstance(insights, dict):
            return "[ERROR] 'insights' must be a JSON object."
        retrieved = data.get("retrieved_chunks", [])
        output_path = data.get("output_path", "tutoring_insights.pdf")
        return f"[INFO] PDF saved: {_save_tutoring_insights_pdf_core(insights, retrieved, output_path)}"
    except json.JSONDecodeError:
        return "[ERROR] Payload must be valid JSON."
    except Exception as e:
        return f"[ERROR] Failed to save PDF: {e}"

# ----------------- Agent -----------------
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

AGENT_SYSTEM_PROMPT = (
    DEFAULT_SYSTEM_PROMPT
    + "\n\nProcess:\n"
    "1) First call the `rag_search` tool with the user's query to fetch transcript excerpts.\n"
    "2) Use those excerpts to produce insights that strictly match the requested format.\n"
    "3) If the user asks to save as PDF, call `save_tutoring_insights_pdf` with a JSON payload containing the insights, "
    "retrieved_chunks (optional), and output_path (optional).\n"
    "Return only the formatted insights unless explicitly asked to save."
)

# (A) Build the parser for your schema
parser = PydanticOutputParser(pydantic_object=TutoringInsights)

# (B) IMPORTANT: Inject parser's format contract into the system message
prompt = ChatPromptTemplate.from_messages([
    ("system", AGENT_SYSTEM_PROMPT + "\n{format_instructions}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
]).partial(format_instructions=parser.get_format_instructions())

load_dotenv()
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), openai_api_key=os.getenv("openai_api_key"), temperature=0)

tools = [rag_search, save_tutoring_insights_pdf]
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)
agentexecutor = AgentExecutor(agent=agent, tools=tools, verbose=True)



In [8]:
# ----------------- Run agent -----------------
query = (
  "Assess the student's understanding while math and science tutoring and what to do next. "
  "Then save the insights as a PDF named 'student_insights.pdf'."
)
result = agentexecutor.invoke({"input": query})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `rag_search` with `{'query': 'math and science tutoring'}`


[INFO] Querying vector store for: 'math and science tutoring'
{"role": "assistant", "text": "Hello there! Welcome to today's math tutoring session! We're going to learn about solving a word problem. Would you like me to read the problem out loud for you?"}

{"role": "assistant", "text": "Hey there! Welcome to today's science tutoring session! Today, we\u2019re going to learn about the states of matter. Would you like me to start by explaining what they are?"}

{"role": "assistant", "text": "That's right! Now, the next question: What operation should we use to find the total number of pencils needed for all students?"}

{"role": "assistant", "text": "That's right! Now, the next question: What operation should we use to find the total number of pencils each student has after getting the extra pencils?"}

{"role": "assistant", "text": "That's right! We'll divide to find out how 